In [ ]:
# Import modules
import paramiko
import os
from stat import S_ISDIR
import pandas as pd
import dotenv
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from pprint import pprint
import json
from src import utils

In [ ]:
# Provide connection details in separate .env file for security
dotenv.load_dotenv()
hostname = os.getenv('HOSTNAME')  # Replace with your server's IP or hostname
user = os.getenv('USER')    # Replace with your SSH username    
password = os.getenv('PASSWORD')  # SSH password from .env file

PROJECT_DIRECTORY = '/mnt/data_raid/feierabend/AVL_FIRE/PEMWE/PEMStar_2'    # Project directory on the remote server
MODEL_NAME = 'PEMStar_BekaertPTL'                   # AVL FIRE model name
CASE_SET_NAME = 'PolCurve_Bek~rtPTL_Update'        # Case set name within the model
CASE_NAME = None                                    # Set to None to search all cases, or specify a case name like 'Case_1'

# Or, if you want to search for files with specific names
# TARGET_FILENAMES = ['file1.txt', 'report.log']


In [ ]:
# --- SSH Client Initialization ---
ssh_client = paramiko.SSHClient()
# Automatically add the server's host key. For production, it's better to manage known_hosts explicitly.
ssh_client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    # Connect to the SSH server
    ssh_client.connect(hostname=hostname, username=user, password=password)
    print(f"Connected to {hostname} using password.")

    # Open an SFTP client
    sftp_client = ssh_client.open_sftp()
    print(f"Opened SFTP session.")
except paramiko.AuthenticationException:
    print("Authentication failed. Check your username, password, or private key.")
except paramiko.SSHException as e:
    print(f"Could not establish SSH connection: {e}")  

In [ ]:
# Define function for retrieving remote data files from AVL FIRE model cases

    

# Retrieve Simulation Input Data
Get input from ".asix" file

In [ ]:
input_data_paths = utils.retrieve_avl_fire_data_paths(
    sftp_client=sftp_client,
    project_directory=PROJECT_DIRECTORY,
    model_name=MODEL_NAME,
    case_set_name=CASE_SET_NAME,
    data_directory='input',
    file_extension='.asix'
)

In [ ]:
from src.asix_parser import parse_asix
input_data_dicts_list = []
for data_path in input_data_paths:
    with sftp_client.open(data_path, 'r') as data_file:
        # data = remote_file.read()
        data = parse_asix(data_file, always_list=False, keep_all_attributes=True, cast_values=True)
        input_data_dicts_list.append(data)
    # data_dicts
input_data_dicts_list

In [ ]:
input_data = input_data_dicts_list[0]

In [ ]:
# from src.utils import _json_default
# with open('testing_output_v3.json', 'w', encoding='utf-8') as f:
#     json.dump(data_dicts, f, default=_json_default, indent=2)

# Retrieve Simulation Input Data
Get input from ".asix" file

In [ ]:
results_2d_data_paths = utils.retrieve_avl_fire_data_paths(
    sftp_client=sftp_client,
    project_directory=PROJECT_DIRECTORY,
    model_name=MODEL_NAME,
    case_set_name=CASE_SET_NAME,
    data_directory='results',
    file_extension='.csv'
)
print(results_2d_data_paths)

In [ ]:
data_path = results_2d_data_paths[0]
result_2d_result_list = []
for data_path in results_2d_data_paths:
    with sftp_client.open(data_path, 'r') as data_file:
        df = pd.read_csv(data_file, header=[1,2], sep=';')  # Adjust separator if needed
        result_2d_result_list.append(df)
result_2d = result_2d_result_list[0]

In [ ]:
results_monitoring_data_paths = utils.retrieve_avl_fire_data_paths(
    sftp_client=sftp_client,
    project_directory=PROJECT_DIRECTORY,
    model_name=MODEL_NAME,
    case_set_name=CASE_SET_NAME,
    data_directory='results',
    file_extension='_flc.csv'
)
print(results_monitoring_data_paths)

In [ ]:
data_path = results_monitoring_data_paths[0]
result_monitoring_result_list = []
for data_path in results_monitoring_data_paths:
    with sftp_client.open(data_path, 'r') as data_file:
        df = pd.read_csv(data_file, header=[1,2], sep=';')  # Adjust separator if needed
        result_monitoring_result_list.append(df)
result_monitoring_result_list[0].head()

In [ ]:
# Close the connections
if 'sftp_client' in locals() and sftp_client:
    sftp_client.close()
    print("SFTP session closed.")
if 'ssh_client' in locals() and ssh_client:
    ssh_client.close()
    print("SSH connection closed.")

In [ ]:
import importlib
import src.firem_name_parser_integration as firem_parser
importlib.reload(firem_parser)
from src.firem_name_parser_integration import (
    normalize_2d_results_columns,
    rename_2d_results_columns,
    normalize_case_bundle,
    load_yaml_from_github
)
# from src.firem_name_parser_integration import load_yaml_from_github

rules_path = load_yaml_from_github()

# Single case
mapping_df = normalize_2d_results_columns(result_2d, input_data, rules_path)
df_2d_renamed, rename_map = rename_2d_results_columns(result_2d, input_data, rules_path)

# All cases
renamed_data_list = []
for i in range(len(result_2d_result_list)):
    renamed_data, _ = rename_2d_results_columns(result_2d_result_list[i], input_data_dicts_list[i], rules_path)
    renamed_data_list.append(renamed_data)

In [ ]:
current_density_key = 'current_density__mea_flow_channels'
cell_voltage_key = 'cell_voltage__mea_flow_channels'
matplotlib.rcParams.update({'font.size': 12})

current_density = []
cell_voltage = []
for df in renamed_data_list:
    if current_density_key not in df.columns and cell_voltage_key not in df.columns:
        raise KeyError(f"Keys not found in DataFrame with columns: {df.columns}")
    
    current_density.append(df[current_density_key].iloc[-1])
    cell_voltage.append(df[cell_voltage_key].iloc[-1])
current_density = np.asarray(current_density)
cell_voltage = np.asarray(cell_voltage)
simulation_data = np.stack((current_density / 10000, cell_voltage))  # Sort by current_density
simulation_data.sort()


In [ ]:
measurement_data = pd.read_csv('reference_data\pem_west_measurement_70C.csv')


In [ ]:
fig, ax = plt.subplots()
ax.plot(measurement_data['current_density_A_cm-2'], measurement_data['voltage_V'], marker='x', label='Measurement', linestyle='-', color='red')
ax.plot(simulation_data[0], simulation_data[1], marker='o', label='CFD Simulations', linestyle='-', color='blue')
ax.set_xlabel('Current Density / A/cm²')
ax.set_ylabel('Cell Voltage / V')
ax.legend()

In [ ]:
simulation_data[0]

In [ ]:
renamed_data

In [ ]:
input_data_jsons = [json.dumps(input_dict, default=utils.json_default) for input_dict in input_data_dicts_list]

In [ ]:
input_data_jsons[0]

In [ ]:
with open("input_file.json", "w", encoding="utf-8") as f:
    json.dump(
        input_data_dicts_list[0],
        f,
        default=utils.json_default,
        indent=2,
        ensure_ascii=False,
    )